In [1]:
import math

import numpy as np
import pandas as pd
import torch
from kan import KAN
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [2]:
data = pd.read_csv("DATA/income (IC)/Preprocessed/data_processed.csv")

y = data["target_label"].values
data.drop("target_label", axis=1, inplace=True)
X = data.values.astype(np.float32)

if y.dtype == "object":
    label_encoder = LabelEncoder()
    y = torch.tensor(label_encoder.fit_transform(y)).reshape(-1, 1).float()
else:
    y = torch.tensor(y).reshape(-1, 1).float()

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


pca = PCA(n_components=0.999)
X_train = torch.tensor(pca.fit_transform(X_train))
X_test = torch.tensor(pca.transform(X_test))

input_shape = X_train.shape[1]

print('Reduce number of features from %d to %d.' % (data.shape[1], input_shape))

dataset = {
    "train_input": X_train,
    "train_label": y_train,
    "test_input": X_test,
    "test_label": y_test,
}

Reduce number of features from 108 to 2.


In [4]:
model = KAN([input_shape, 5, 1]).speed()
history = model.fit(dataset, steps=20)
y_score = model(X_test)
y_pred = (y_score > 0.5).int()

checkpoint directory created: ./model
saving model version 0.0


| train_loss: 4.91e-01 | test_loss: 4.91e-01 | reg: 0.00e+00 | : 100%|█| 20/20 [00:27<00:00,  1.36s/


In [5]:
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print('ROC-AUC: %.4f' % roc_auc_score(y_test, y_pred))

              precision    recall  f1-score   support

          No       0.81      0.23      0.36      4950
         Yes       0.55      0.95      0.70      4938

    accuracy                           0.59      9888
   macro avg       0.68      0.59      0.53      9888
weighted avg       0.68      0.59      0.53      9888

ROC-AUC: 0.5876
